# 00 — Generate Synthetic Data

Goal: create a synthetic binary-classification dataset that mimics the target experiment scale
(**10–20M rows x 50 features**), with a *known* logistic data-generating process (DGP), so we can
later judge how close each model's predictions/coefficients get to the ground truth.

**Design choices**

- Features `X ~ N(0, 1)`, i.i.d., float32 (halves memory vs float64; at 20M x 50 that's the
  difference between ~4GB and ~8GB for `X` alone), generated **directly in float32** (via
  `Generator.standard_normal(..., dtype=np.float32)`) rather than generated as float64 and cast
  down, which would otherwise allocate — and briefly hold — both copies at once.
- True linear index `logit = X @ true_beta + intercept`, plus an unobserved noise term (so the
  Bayes error rate isn't zero — otherwise every model looks perfect).
- Labels drawn as `y ~ Bernoulli(sigmoid(logit + noise))` — this is exactly the model
  **logistic regression** assumes, and only an *approximation* of what a **linear probability
  model (LPM)** assumes (LPM assumes `E[y|X]` is linear in X, not sigmoid-linear) — that
  mismatch is intentional, it's what you'd see with real data too.
- Data is generated **chunk by chunk and written straight into the final `.npy` file with plain
  buffered file I/O** (`seek` + `write`, then `flush`/`fsync`) — deliberately **not** through a
  memory-mapped array. A memory map keeps every page you've written resident in *your own
  process's* RSS for as long as the mapping is open (flushing only makes those pages "clean," it
  doesn't release them) — on a constrained container that RSS growth can itself trigger an OOM
  kill even though the underlying memory is technically page-cache and reclaimable. Plain
  `write()` calls go through the OS page cache without ever being mapped into this process's
  address space, so this notebook's own memory use stays flat as `N_ROWS` grows — watch the
  `RSS=` numbers printed during generation to see this for yourself.

Run this notebook once per `N_ROWS` value you want to test; it caches the arrays under `data/`
so `01_...` and `02_...` can just load them.

In [1]:
import gc
import math
import os
import time
from pathlib import Path

import numpy as np
import psutil

# ---------------------------------------------------------------------------
# CONFIG — every knob for this notebook lives here. Each can also be set from
# outside the notebook (e.g. on the GCP Workbench, or via `papermill`/
# `jupyter nbconvert --execute`) by exporting the matching env var before
# launching Jupyter, so you don't have to hand-edit the notebook per run:
#   N_ROWS=10000000 N_FEATURES=50 jupyter nbconvert --execute --to notebook \
#       00_generate_synthetic_data.ipynb
#
# Smoke-test defaults are small so this notebook runs in seconds as shipped.
# For the real GCP Workbench experiment, set N_ROWS to 10_000_000 or
# 20_000_000. Rules of thumb for RAM budget (float32 X only, 50 features):
#   10,000,000 rows -> ~2.0 GB for X   (comfortable on 32GB)
#   20,000,000 rows -> ~4.0 GB for X   (comfortable on 32GB, very safe on 64GB)
# sklearn will make additional working copies during fit/split, so budget
# 3-5x the raw array size as a rough safety margin — that's a training-time
# concern though (notebook 01/02); THIS notebook's own memory use should stay
# close to flat regardless of N_ROWS (watch the RSS= numbers it prints).
# ---------------------------------------------------------------------------
N_ROWS = int(os.environ.get("N_ROWS", 200_000))            # <-- set to 10_000_000 / 20_000_000 for the real run
N_FEATURES = int(os.environ.get("N_FEATURES", 50))
RANDOM_STATE = int(os.environ.get("RANDOM_STATE", 42))
CHUNK_SIZE = int(os.environ.get("CHUNK_SIZE", 200_000))     # rows generated per chunk; bounds peak RAM during generation
TRUE_BETA_SCALE = float(os.environ.get("TRUE_BETA_SCALE", 0.5))  # spread of the ground-truth coefficients
NOISE_SCALE = float(os.environ.get("NOISE_SCALE", 1.0))          # unobserved-noise std dev (controls Bayes error)
TRUE_INTERCEPT = float(os.environ.get("TRUE_INTERCEPT", -0.3))
FORCE_REGENERATE = os.environ.get("FORCE_REGENERATE", "0") == "1"  # set to "1" to overwrite cached data

DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))
DATA_DIR.mkdir(exist_ok=True)
X_PATH = DATA_DIR / f"X_{N_ROWS}.npy"
Y_PATH = DATA_DIR / f"y_{N_ROWS}.npy"

print(f"N_ROWS={N_ROWS:,}  N_FEATURES={N_FEATURES}  CHUNK_SIZE={CHUNK_SIZE:,}")
print(f"Estimated X size on disk: {N_ROWS * N_FEATURES * 4 / 1e9:.2f} GB (float32)")
print(f"Available system RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")

N_ROWS=200,000  N_FEATURES=50  CHUNK_SIZE=200,000
Estimated X size on disk: 0.04 GB (float32)
Available system RAM: 17.2 GB


In [2]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def generate_dataset(n_rows, n_features, x_path, y_path, chunk_size, random_state,
                      beta_scale, noise_scale, intercept, verbose=True):
    """Writes X (n_rows x n_features, float32) and y (n_rows, float32) chunk by
    chunk directly into valid .npy files on disk, using plain seek+write (not a
    memory-mapped array) so the data is never resident in THIS process's own
    RSS -- only in the OS page cache, which is trivially reclaimable. Flushes
    and fsyncs after every chunk so dirty pages never build up beyond one
    chunk's worth either. This is what actually generates the data used by
    notebooks 01 and 02 -- nothing downstream regenerates it."""
    rng = np.random.default_rng(random_state)
    process = psutil.Process(os.getpid())

    # Ground-truth coefficients for the logistic DGP. Kept small-ish (beta_scale)
    # so class probabilities aren't saturated near 0/1 for every row.
    true_beta = rng.standard_normal(n_features, dtype=np.float32) * np.float32(beta_scale)
    true_intercept = np.float32(intercept)

    # Create correctly-headered, correctly-sized .npy files up front. This is
    # cheap -- numpy just writes a small header then truncates the file to its
    # final length, it does not touch/zero the data region. We immediately
    # discard these memmap objects and never write through them; they only
    # exist to get a valid header + the exact byte offset the raw data starts
    # at (`.offset`), which plain file I/O below writes into directly.
    x_mm_init = np.lib.format.open_memmap(x_path, mode="w+", dtype=np.float32, shape=(n_rows, n_features))
    x_offset, x_row_nbytes = x_mm_init.offset, n_features * x_mm_init.dtype.itemsize
    del x_mm_init

    y_mm_init = np.lib.format.open_memmap(y_path, mode="w+", dtype=np.float32, shape=(n_rows,))
    y_offset, y_row_nbytes = y_mm_init.offset, y_mm_init.dtype.itemsize
    del y_mm_init

    n_chunks = math.ceil(n_rows / chunk_size)
    log_every = max(1, n_chunks // 20)  # ~20 progress lines regardless of scale

    t0 = time.perf_counter()
    with open(x_path, "r+b") as xf, open(y_path, "r+b") as yf:
        for i, start in enumerate(range(0, n_rows, chunk_size)):
            end = min(start + chunk_size, n_rows)
            n = end - start

            # Generate directly in float32 -- avoids the float64 intermediate
            # array (and the copy on .astype) that rng.normal(...).astype(f32)
            # would otherwise allocate for every chunk.
            Xc = rng.standard_normal(size=(n, n_features), dtype=np.float32)
            logits = Xc @ true_beta + true_intercept
            noise = rng.standard_normal(size=n, dtype=np.float32) * np.float32(noise_scale)
            p = sigmoid(logits + noise)
            u = rng.random(size=n, dtype=np.float32)
            yc = (u < p).astype(np.float32)

            xf.seek(x_offset + start * x_row_nbytes)
            xf.write(Xc.tobytes())
            yf.seek(y_offset + start * y_row_nbytes)
            yf.write(yc.tobytes())

            # Force this chunk to disk now, rather than letting dirty pages
            # accumulate until the loop finishes.
            xf.flush(); os.fsync(xf.fileno())
            yf.flush(); os.fsync(yf.fileno())

            del Xc, logits, noise, p, u, yc
            gc.collect()

            if verbose and (i % log_every == 0 or end == n_rows):
                rss_gb = process.memory_info().rss / 1e9
                print(f"  chunk {i + 1}/{n_chunks}: {end:,}/{n_rows:,} rows written, RSS={rss_gb:.2f} GB")

    elapsed = time.perf_counter() - t0

    # Reopen read-only for the sanity checks below and for downstream notebooks.
    X_mm = np.load(x_path, mmap_mode="r")
    y_mm = np.load(y_path, mmap_mode="r")
    return X_mm, y_mm, true_beta, true_intercept, elapsed


if X_PATH.exists() and Y_PATH.exists() and not FORCE_REGENERATE:
    print(f"Found cached data at {X_PATH} / {Y_PATH}, skipping generation.")
    print("Set FORCE_REGENERATE=1 (or change N_ROWS) to regenerate.")
    X_mm = np.load(X_PATH, mmap_mode="r")
    y_mm = np.load(Y_PATH, mmap_mode="r")
else:
    X_mm, y_mm, true_beta, true_intercept, elapsed = generate_dataset(
        N_ROWS, N_FEATURES, X_PATH, Y_PATH, CHUNK_SIZE, RANDOM_STATE,
        TRUE_BETA_SCALE, NOISE_SCALE, TRUE_INTERCEPT,
    )
    np.save(DATA_DIR / f"true_beta_{N_ROWS}.npy", true_beta)
    print(f"Generated {N_ROWS:,} rows x {N_FEATURES} features in {elapsed:.2f}s")
    print(f"X: {X_PATH}  ({X_PATH.stat().st_size / 1e9:.2f} GB)")
    print(f"y: {Y_PATH}  ({Y_PATH.stat().st_size / 1e9:.2f} GB)")

  chunk 1/1: 200,000/200,000 rows written, RSS=0.17 GB
Generated 200,000 rows x 50 features in 0.08s
X: data/X_200000.npy  (0.04 GB)
y: data/y_200000.npy  (0.00 GB)


In [3]:
# Sanity checks
print("X shape:", X_mm.shape, X_mm.dtype)
print("y shape:", y_mm.shape, y_mm.dtype)
print(f"Positive class rate: {y_mm.mean():.3f}")
print(f"X mean/std (should be ~0/~1): {X_mm[:50_000].mean():.3f} / {X_mm[:50_000].std():.3f}")

X shape: (200000, 50) float32
y shape: (200000,) float32
Positive class rate: 0.472
X mean/std (should be ~0/~1): -0.000 / 1.000


## Notes for scaling this up on the GCP Workbench

- **32GB machine**: comfortably handles 10M rows. 20M rows is workable but leaves less headroom
  for the batch `LogisticRegression` solver's internal copies during `train_test_split` — prefer
  `SGDClassifier` (partial_fit / streaming) if you hit `MemoryError`, or drop to float32
  throughout (already done here) and avoid `pandas.DataFrame` wrappers (keep everything as raw
  numpy arrays). Note this headroom concern is about **notebook 01/02 (training)**, not this
  notebook — data *generation* here should use only a few hundred MB regardless of `N_ROWS`.
- **64GB machine**: comfortably handles 20M rows with all three models, including the full-batch
  `LogisticRegression(solver="lbfgs")` baseline.
- Re-run this notebook once per scale you want to test (e.g. `N_ROWS = 1_000_000`, then
  `10_000_000`, then `20_000_000`) — each size is cached under `data/` with the row count in the
  filename, so `01_...` and `02_...` just need `N_ROWS` set to match.

### If the kernel still dies during *generation* at 10-20M rows

This generator writes through plain file I/O (not a memory map) specifically so its own memory
use stays flat — watch the `RSS=` numbers it prints; they should hover in the low hundreds of MB
regardless of `N_ROWS`, not climb toward the full dataset size. If RSS does stay flat and the
kernel *still* dies, the ceiling isn't this notebook's Python-level memory use — it's something at
the container/VM level:

1. **Check the actual memory limit the kernel process is subject to** — on a GCP Workbench
   instance the Jupyter kernel sometimes runs inside a container with a lower limit than the
   VM's advertised RAM. From a terminal (or a `!`-prefixed cell): `free -h` for what the OS sees,
   and `cat /sys/fs/cgroup/memory.max` (cgroup v2) or
   `cat /sys/fs/cgroup/memory/memory.limit_in_bytes` (cgroup v1) for the container's actual cap.
2. **Confirm the OOM killer is the culprit** — `dmesg | egrep -i 'killed process|out of memory'`
   (or `journalctl -k | grep -i oom`) right after a crash will show a `Killed process <pid>
   (python...)` line if so, versus a Jupyter-side timeout/disconnect, which needs a different fix.
3. **Confirm `data/` is on real persistent disk, not tmpfs** — run `df -h data/` and
   `mount | grep "$(df --output=target data/ | tail -1)"`. If the mount type is `tmpfs`, every
   byte "written" to disk is actually consuming RAM regardless of how it's written — point
   `DATA_DIR` at a path on the persistent disk instead (e.g. `DATA_DIR=/home/jupyter/data`).
4. **Still tight?** Lower `CHUNK_SIZE` further (e.g. `CHUNK_SIZE=50000`) — smaller chunks trade
   more (cheap) flush/fsync calls for an even smaller peak working set.